# 01 Filter and Clean Text

Takes `raw_data_all/all_data.csv` and:

1. Removes rows marked for exclusion (`wiki_flag` is TRUE or `wiki_rule` is not blank). Removed rows are saved with the reason in `processed/01_removed.csv`.
2. Cleans the text: unicode normalised, curly quotes made straight, line breaks and repeated spaces collapsed, lowercased.

Output: `processed/01_filtered.csv`. `row_id` is the row number in `all_data.csv` and follows each answer through every later step.

In [1]:
import pandas as pd
import unicodedata
import re
from pathlib import Path

Path("processed").mkdir(exist_ok=True)

In [2]:
data = pd.read_csv("raw_data_all/all_data.csv", dtype=str, keep_default_na=False)
data.insert(0, "row_id", range(1, len(data) + 1))
print(f"{len(data):,} rows in all_data.csv")

190,400 rows in all_data.csv


## Remove flagged rows

In [3]:
exclude = (data["wiki_flag"] == "True") | (data["wiki_rule"] != "")

removed = data[exclude].copy()
removed["reason"] = removed.apply(
    lambda r: ",".join(filter(None, ["original_cleaning" if r["wiki_flag"] == "True" else "", r["wiki_rule"]])),
    axis=1,
)
removed[["row_id", "source_file", "subject_id", "word", "answer", "reason"]].to_csv(
    "processed/01_removed.csv", index=False
)

kept = data[~exclude].copy()
print(f"Removed {len(removed):,} rows, kept {len(kept):,}")
print(removed["reason"].str.split(",").explode().value_counts())

Removed 4,906 rows, kept 185,494
reason
original_cleaning       3368
semicolon_definition     718
rejected_hit             541
numbered_senses          351
long_duplicate           336
very_long                326
article_text             120
pos_tag                   88
web_text                  68
stray_cue                 29
Name: count, dtype: int64


## Clean the text

In [4]:
QUOTES = {"\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'", "`": "'", "\u00b4": "'",
          "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u2013": "-", "\u2014": "-"}

def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\\'", "'").replace('\\"', '"')  # escaped quotes left over from the export
    text = re.sub(r"[\x00-\x08\x0b-\x1f\x7f]", " ", text)  # stray control characters
    for old, new in QUOTES.items():
        text = text.replace(old, new)
    text = re.sub(r"\s+", " ", text)  # line breaks, tabs, repeated spaces
    return text.strip().lower()

kept["answer_clean"] = kept["answer"].map(clean_text)
kept["cue"] = kept["word"].map(clean_text)

empty = kept["answer_clean"] == ""
print(f"{empty.sum()} answers empty after cleaning (dropped)")
kept = kept[~empty]

0 answers empty after cleaning (dropped)


In [5]:
out = kept[["row_id", "subject_id", "file_subject_id", "source_file", "cue", "answer", "answer_clean"]]
out = out.rename(columns={"answer": "answer_raw"})
out.to_csv("processed/01_filtered.csv", index=False)
print(f"Saved {len(out):,} rows to processed/01_filtered.csv")
out.sample(5, random_state=1)

Saved 185,494 rows to processed/01_filtered.csv


,row_id,subject_id,file_subject_id,source_file,cue,answer_raw,answer_clean
165052,165053,12796,463,word_long_all.csv,hay,dry grasses,dry grasses
3172,3173,54,54,individual_subjects.csv,forceful,teacher,teacher
44920,44921,738,739,individual_subjects.csv,tree,"plant, outdoors, big and tall, part of nature","plant, outdoors, big and tall, part of nature"
112925,112926,6854,5044,word_list_5_long.csv,intoxicated,"drunk, inebriated","drunk, inebriated"
133726,133727,11017,9207,word_list_5_long.csv,surface,"the outside of an object. it can be hard, soft...","the outside of an object. it can be hard, soft..."
